In [17]:
!pip install --upgrade langchain langchain-core langchain-community pydantic
!pip install langchain-openai
!pip install -U faiss-cpu
!pip install pypdf
!pip install openai
!pip install python-dotenv
!pip install docx2txt
!pip install gradio
!pip install tiktoken


In [18]:
import os
import gradio as gr
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.chains import LLMChain
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader, TextLoader
from langchain_community.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.runnables import RunnableLambda, RunnableMap
import re
import tempfile
import pickle
from typing import Optional
import shutil
from getpass import getpass

In [19]:
from dotenv import load_dotenv
load_dotenv()

True

Initialize Components

In [20]:
# Setup embedding model
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=os.getenv("OPENAI_API_KEY")
)

 Initialize or load FAISS vector store

In [21]:
VECTOR_STORE_DIR = "/content/faiss_index"
VECTOR_STORE_INDEX = os.path.join(VECTOR_STORE_DIR, "index.faiss")
VECTOR_STORE_PKL = os.path.join(VECTOR_STORE_DIR, "index.pkl")

In [22]:
def initialize_vector_store():
    """Initialize or load FAISS vector store using FAISS native methods"""
    global vectorstore

    # Create directory if it doesn't exist
    if not os.path.exists(VECTOR_STORE_DIR):
        os.makedirs(VECTOR_STORE_DIR)

    # Check if existing store exists
    if os.path.exists(VECTOR_STORE_INDEX) and os.path.exists(VECTOR_STORE_PKL):
        try:
            # Load existing FAISS store using native method
            vectorstore = FAISS.load_local(
                VECTOR_STORE_DIR,
                embedding_model,
                index_name="index",
                allow_dangerous_deserialization=True  # Required for loading
            )
            print("Loaded existing FAISS vector store")
        except Exception as e:
            print(f"Error loading existing store: {e}")
            # Create new if loading fails
            from langchain.docstore.document import Document
            dummy_doc = Document(page_content="Initial document", metadata={"source": "init"})
            vectorstore = FAISS.from_documents([dummy_doc], embedding_model)
            print("Created new FAISS vector store after load error")
    else:
        # Create new FAISS store
        from langchain.docstore.document import Document
        dummy_doc = Document(page_content="Initial document", metadata={"source": "init"})
        vectorstore = FAISS.from_documents([dummy_doc], embedding_model)
        # Save it immediately
        vectorstore.save_local(VECTOR_STORE_DIR, index_name="index")
        print("Created and saved new FAISS vector store")

    return vectorstore

In [23]:
vectorstore = initialize_vector_store()

Loaded existing FAISS vector store


Document Processing Functions

In [24]:
def extract_text_from_file(file_path):
    """Extract text from uploaded legal files"""
    file_extension = os.path.splitext(file_path)[1].lower()

    try:
        if file_extension == '.pdf':
            loader = PyPDFLoader(file_path)
        elif file_extension == '.docx':
            loader = Docx2txtLoader(file_path)
        elif file_extension == '.txt':
            loader = TextLoader(file_path)
        else:
            raise ValueError(f"Unsupported file format: {file_extension}")

        documents = loader.load()
        text = " ".join([doc.page_content for doc in documents])
        return text
    except Exception as e:
        raise Exception(f"Error extracting text: {str(e)}")

In [25]:
def split_text(text):
    """Split text into chunks for processing"""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    return splitter.create_documents([text])

In [26]:
def store_document_analysis(output_text, analysis, document_name):
    """Store resume analysis in FAISS vector store using native save method"""
    global vectorstore

    try:
        # Create metadata for the document
        metadata = {
            "document_name": document_name,
            "type": "legal_analysis",
            "full_analysis": analysis[:1000]  # Store first 1000 chars in metadata
        }

        # Split the analysis into chunks
        documents = split_text(f"Document: {output_text}\n\nAnalysis: {analysis}")

        # Add metadata to each document
        for doc in documents:
            doc.metadata = metadata

        # Add to FAISS store
        vectorstore.add_documents(documents)

        # Save using FAISS native method instead of pickle
        vectorstore.save_local(VECTOR_STORE_DIR, index_name="index")

        return True

    except Exception as e:
        print(f"Error storing analysis: {str(e)}")
        return False

Main Analysis Function

In [27]:
def analyze_document(prompt, legal_file):
    """Main function to analyze legal document for given question"""

    if not prompt or not legal_file:
        return "Please provide both proompt and a legal file.", None, None

    try:
        # Extract text from resume
        output_text = extract_text_from_file(legal_file.name)

        # Initialize OpenAI LLM
        llm = ChatOpenAI(
            model="gpt-4",  # You can also use "gpt-3.5-turbo" for lower cost
            openai_api_key=os.getenv("OPENAI_API_KEY"),
            temperature=0.2,
            max_tokens=1500
        )

        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["prompt", "output_text"],
            template="""
            You are an expert in Legal Professional. Analyze the document below against the prompt.

            prompt:
            {prompt}

            output_text:
            {output_text}

            Provide a comprehensive answer

            """
        )

        # Create and run the chain using LCEL
        chain = (
            RunnableMap({
                "prompt": lambda x: x["prompt"],
                "output_text": lambda x: x["output_text"]
            })
            | prompt_template
            | llm
            | StrOutputParser()
        )

        # Run the analysis
        analysis = chain.invoke({
            "prompt": prompt,
            "output_text": output_text
        })

        # Store in FAISS vector store
        store_document_analysis(output_text, analysis, legal_file.name)

        # Create downloadable content
        download_content = f"""
Legal Document Report
================================
Legal Advice Required:
----------------
{prompt}

AI ANALYSIS:
-----------
{analysis}

================================
        """

        return analysis

    except Exception as e:
        return f"Error during analysis: {str(e)}", None, None

In [28]:
def analyze_Query(prompt):
    """Main function to analyze Legal Query for given question"""

    if not prompt:
        return "Please provide  proompt for Legal Query", None, None

    try:
        # Initialize OpenAI LLM
        llm = ChatOpenAI(
            model="gpt-4",  # You can also use "gpt-3.5-turbo" for lower cost
            openai_api_key=os.getenv("OPENAI_API_KEY"),
            temperature=0.2,
            max_tokens=1500
        )

        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["prompt"],
            template="""
            You are an expert in Legal Professional.Reply in appropriate summary format

            prompt:
            {prompt}

            Provide a appropriate answer

            """
        )

        # Create and run the chain using LCEL
        chain = (
            RunnableMap({
                "prompt": lambda x: x["prompt"]
            })
            | prompt_template
            | llm
            | StrOutputParser()
        )

        # Run the analysis
        analysis = chain.invoke({
            "prompt": prompt
        })

        # Create downloadable content
        download_content = f"""
Legal Document Report
================================
Legal Advice Required:
----------------
{prompt}

AI ANALYSIS:
-----------
{analysis}

================================
        """

        return analysis

    except Exception as e:
        return f"Error during analysis: {str(e)}", None, None

In [29]:
def create_gradio_interface2():
    """Create Gradio interface for the application"""

    with gr.Blocks(title="Legal Advice with FAISS & OpenAI", theme=gr.themes.Soft()) as demo:

        gr.Markdown("""
        # 📋 Legal Advice Screening Application
        ### Powered by FAISS Vector Store and OpenAI GPT-4

        Upload a legal document and provide query to get an AI-powered analysis with the document.
        """)

        with gr.Tab("Legal Advice Analysis"):
            with gr.Row():
                with gr.Column():
                    job_req_input = gr.Textbox(
                        label="Legal Query",
                        placeholder="Enter the legal query.",
                        lines=10
                    )

                with gr.Column():
                    resume_upload = gr.File(
                        label="Upload document",
                        file_types=[".pdf", ".docx", ".txt"],
                        type="filepath"
                    )

            # Two buttons instead of one
            with gr.Row():
                analyze_btn = gr.Button("🔍 Analyze Legal Document", variant="primary")
                download_btn = gr.Button("🔍 Analyze Legal Query", variant="primary")

            with gr.Row():
                analysis_output = gr.Markdown(label="Analysis Results")

            with gr.Row():
                download_text = gr.Textbox(
                    label="Analysis Report (Copy or Download)",
                    lines=10,
                    visible=False
                )

            # First button: analyze
            analyze_btn.click(
                fn=analyze_document,
                inputs=[job_req_input, resume_upload],
                outputs=[analysis_output]
            )

            # Second button: show report text
            def show_report(result_text):
                if result_text:
                    return gr.update(value=result_text, visible=True)
                else:
                    return gr.update(visible=False)

            download_btn.click(
                fn=analyze_Query,
                inputs=[job_req_input],
                outputs=[analysis_output]
            )

        gr.Markdown("""
        ---
        ### 📝 Instructions:
        1. Enter detailed legal query in the left panel
        2. Upload a document (File in PDF format)
        3. Click "Analyze Legal Document" to get AI-powered insights
        4. Click "Analyze Legal Query to get AI-powered Summary

        ### 💾 Data Storage:
        - All analyses are stored in a FAISS vector database for future reference
        - You can search through previously analyzed detail using semantic search
        """)

    return demo


In [30]:
demo = create_gradio_interface2()
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d7b21b43b68eeff54c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
